In [ ]:
%pip install --upgrade langchain langchain-core langchain-community pydantic duckduckgo-search langchain_experimental ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


# search tool

In [8]:
from langchain_community.tools import DuckDuckGoSearchResults
search_tool = DuckDuckGoSearchResults()

result = search_tool.invoke("top news in india today")
print(result)

snippet: 4 hours ago - Evening news wrap: Ram Temple scam deepens; Prashant Kishor to enter politics; Mojtaba skips father’s funeral · The long journey to your tap: How India's biggest cities get their water · New York based artist Jeena Raghavan presents new solo exhibition in Bangalore · Mumbai rain chaos: Flights hit, roads flooded, daily life disrupted — top developments, title: India News | Latest India News Today & Real-time News from India, India Breaking News, link: https://timesofindia.indiatimes.com/india, snippet: 10 hours ago - National news and headlines from within or around India with live updates on major breaking events from The Hindu., title: India Latest News: Top National Headlines Today & Breaking News - The Hindu, link: https://www.thehindu.com/news/national/, snippet: 4 hours ago - Indian Gukesh Dommaraju, 18, becomes the youngest ever chess world champion after Ding Liren blunder ... It’s got tigers, tea plantations and beaches — so why does Bangladesh struggle 

In [11]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_results_json
A wrapper around Duck Duck Go Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


# Shell Tool

In [12]:
from langchain_community.tools import ShellTool

shellTool = ShellTool()

result = shellTool.invoke("ls -a")
print(result)

Executing command:
 ls -a
.
..
.config
sample_data



/usr/local/lib/python3.12/dist-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


In [14]:
print(shellTool.name)
print(shellTool.description)
print(shellTool.args)

terminal
Run shell commands on this Linux machine.
{'commands': {'anyOf': [{'type': 'string'}, {'items': {'type': 'string'}, 'type': 'array'}], 'description': 'List of shell commands to run. Deserialized using json.loads', 'title': 'Commands'}}


# Custom Tools

In [19]:
# using decorator

from langchain_community.tools import tool

@tool
def square(num: int)-> int:
  """return square of the input number."""
  return num**2

result = square.invoke({"num": 4})
print(result)

16


In [20]:
print(square.name)
print(square.description)
print(square.args)

square
return square of the input number.
{'num': {'title': 'Num', 'type': 'integer'}}


In [21]:
# using StructuredTool

from langchain_community.tools import StructuredTool
from pydantic import BaseModel, Field

class Multiplication(BaseModel):
  a: int = Field(description= "First number for multiplication", required=True)
  b: int = Field(description= "Second number for multiplication", required=True)

def multiplication_fun(a: int, b: int) -> int:
  return a * b

multiplication_tool = StructuredTool.from_function(
    name="multiplication_function",
    description= "multiply two numbers",
    func= multiplication_fun,
    args_schema= Multiplication
)

result = multiplication_tool.invoke({"a":9, "b":8})
print(result)

72


/tmp/ipykernel_2005/2793554541.py:7: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(description= "First number for multiplication", required=True)
/tmp/ipykernel_2005/2793554541.py:8: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(description= "Second number for multiplication", required=True)


In [24]:
print(multiplication_tool.name)
print(multiplication_tool.description)
print(multiplication_tool.args)
print(multiplication_tool.args_schema)

multiplication_function
multiply two numbers
{'a': {'description': 'First number for multiplication', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second number for multiplication', 'title': 'B', 'type': 'integer'}}
<class '__main__.Multiplication'>


In [29]:
# using base class

from langchain.tools import BaseTool
from typing import Type

class MultiplicationArgs(BaseModel):
  a: int = Field(description= "First number for multiplication", required=True)
  b: int = Field(description= "Second number for multiplication", required=True)

class MultiplicationTool(BaseTool):
  name: str = "MultiplicationTool"
  description: str = "multiplication tool using basetool class"

  args_schema: Type[BaseModel] = MultiplicationArgs

  def _run(self, a: int, b: int) -> int:
    return a * b

newmultiplicationtool = MultiplicationTool()
result = newmultiplicationtool.invoke({"a":4, "b":6})
print(result)

24


/tmp/ipykernel_2005/2892881886.py:7: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  a: int = Field(description= "First number for multiplication", required=True)
/tmp/ipykernel_2005/2892881886.py:8: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  b: int = Field(description= "Second number for multiplication", required=True)


In [30]:
print(newmultiplicationtool.name)
print(newmultiplicationtool.description)
print(newmultiplicationtool.args)

MultiplicationTool
multiplication tool using basetool class
{'a': {'description': 'First number for multiplication', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second number for multiplication', 'title': 'B', 'type': 'integer'}}
